In [23]:
pip install opencv-python-headless --user


  Using cached opencv_python_headless-4.11.0.86-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python_headless-4.11.0.86-cp37-abi3-win_amd64.whl (39.4 MB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import cv2
from PIL import Image
import numpy as np

In [9]:
from PIL import Image
import os

source_folder = 'D://Thumbs/Bad'
valid_folder = 'D://Thumbs/Bad_valid'

os.makedirs(valid_folder, exist_ok=True)

for file in os.listdir(source_folder):
    if file.endswith('.BMP'):
        path = os.path.join(source_folder, file)
        try:
            with Image.open(path) as img:
                img.verify()  # Check image integrity
            with Image.open(path) as img:
                img.convert('L').save(os.path.join(valid_folder, file.replace('.BMP', '.png')))
        except Exception as e:
            print(f"Skipping {file}: {e}")


Skipping τ_66A3~1.BMP: cannot identify image file 'D:\\Thumbs\\Bad\\τ_66A3~1.BMP'
Skipping Ñ91EA9~5.BMP: cannot identify image file 'D:\\Thumbs\\Bad\\Ñ91EA9~5.BMP'
Skipping 20__M_Right_thumb_finger_Obl.BMP: Unsupported BMP Palette size (268435712)
Skipping 23__M_Right_middle_finger_CR.BMP: image file is truncated (88 bytes not processed)
Skipping 25__F_Left_thumb_finger_Zcut.BMP: image file is truncated (64 bytes not processed)
Skipping 27__M_Left_index_finger_CR.BMP: Unsupported BMP compression (8)
Skipping 27__M_Left_little_finger_CR.BMP: Unsupported BMP pixel depth (12)
Skipping 27__M_Left_little_finger_Obl.BMP: Image size (55297713824 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.
Skipping 27__M_Left_middle_finger_CR.BMP: Unsupported BMP header type (32)
Skipping 27__M_Left_middle_finger_Obl.BMP: Unsupported BMP header type (297)
Skipping 27__M_Left_ring_finger_Obl.BMP: cannot identify image file 'D:\\Thumbs\\Bad\\27__M_Left_ring_finger_Obl.BMP'


In [10]:
from PIL import Image
import os

source_folder = 'D://Thumbs/Good'
valid_folder = 'D://Thumbs/Good_valid'

os.makedirs(valid_folder, exist_ok=True)

for file in os.listdir(source_folder):
    if file.endswith('.BMP'):
        path = os.path.join(source_folder, file)
        try:
            with Image.open(path) as img:
                img.verify()  # Check image integrity
            with Image.open(path) as img:
                img.convert('L').save(os.path.join(valid_folder, file.replace('.BMP', '.png')))
        except Exception as e:
            print(f"Skipping {file}: {e}")


Skipping 1__M_Left_little_finger.BMP: Unsupported BMP compression (524291)
Skipping 1__M_Right_middle_finger.BMP: Unsupported BMP bitfields layout
Skipping 2__F_Left_index_finger.BMP: Unsupported BMP compression (131075)
Skipping 2__F_Left_ring_finger.BMP: Unsupported BMP pixel depth (1056)
Skipping 2__F_Right_little_finger.BMP: cannot identify image file 'D:\\Thumbs\\Good\\2__F_Right_little_finger.BMP'
Skipping 2__F_Right_thumb_finger.BMP: Unsupported BMP bitfields layout
Skipping 17__M_Right_ring_finger.BMP: Unsupported BMP compression (7)
Skipping 17__M_Right_thumb_finger.BMP: Unsupported BMP bitfields layout
Skipping 25__F_Left_little_finger.BMP: Unsupported BMP bitfields layout
Skipping 26__M_Left_ring_finger.BMP: Truncated File Read
Skipping 30__F_Right_index_finger.BMP: Unsupported BMP bitfields layout
Skipping 31__F_Left_little_finger.BMP: Truncated File Read
Skipping 32__M_Right_little_finger.BMP: cannot identify image file 'D:\\Thumbs\\Good\\32__M_Right_little_finger.BMP'
Ski

In [11]:
from sklearn.utils import resample
import os
import random
import shutil

bad_valid = 'D://Thumbs/Bad_valid'
balanced_bad = 'D://Thumbs/Bad_balanced'
os.makedirs(balanced_bad, exist_ok=True)

bad_files = os.listdir(bad_valid)
sampled = random.sample(bad_files, 948)

for f in sampled:
    shutil.copy(os.path.join(bad_valid, f), os.path.join(balanced_bad, f))


In [13]:
path = 'D://Thumbs'
data = []
labels = []
target_size = (100,100)
for folder in (['Bad_balanced', 'Good_valid']):
    filepath = os.path.join(path, folder)
    if folder == 'Bad_balanced':
        label=0
    elif folder == 'Good_valid':
        label = 1
    
    for file in os.listdir(filepath):
        if not file.endswith('.png'):
            print(f"Skipping non-png file: {file}")
            continue

        img_path = os.path.join(filepath, file)

        try:
            img = Image.open(img_path).convert('L')
            img = img.resize(target_size)
            img= np.array(img)/255.0
            if img is None:
                print(f"Skipping unreadable image: {img_path}")
                continue
        except Exception as e:
            print(f"Error reading {img_path}: {e}")
            continue
        
        
        data.append(img.flatten())
        labels.append(label)
        
x = np.array(data)
y= np.array(labels)

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

train_x, test_x, train_y, test_y = train_test_split(x,y, test_size=0.2, random_state=42 )

In [17]:
model = SVC()
model.fit(train_x, train_y)

SVC()

In [18]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(test_x)
print("Accuracy: ",accuracy_score(test_y, y_pred))
print("\n confusion Matrix: ", confusion_matrix(test_y, y_pred))
print("\n classification report: ", classification_report(test_y, y_pred))

Accuracy:  0.7578947368421053

 confusion Matrix:  [[131  62]
 [ 30 157]]

 classification report:                precision    recall  f1-score   support

           0       0.81      0.68      0.74       193
           1       0.72      0.84      0.77       187

    accuracy                           0.76       380
   macro avg       0.77      0.76      0.76       380
weighted avg       0.77      0.76      0.76       380



In [26]:
from sklearn.model_selection import GridSearchCV
param_grid={
    'C': [0.1, 1, 10],
    'kernel':['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.01, 0.001]
}

grid = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')
grid.fit(train_x, train_y)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10],
                         'gamma': ['scale', 'auto', 0.01, 0.001],
                         'kernel': ['linear', 'rbf']},
             scoring='accuracy')

In [27]:
print("best parameters: ", grid.best_params_)
print("\n Best cross-val accuracy", grid.best_score_)
best_model = grid.best_estimator_

best parameters:  {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}

 Best cross-val accuracy 0.7750868507903422


In [28]:
y_pred2 = best_model.predict(test_x)
print("\n classification report: ", classification_report(test_y, y_pred2))


 classification report:                precision    recall  f1-score   support

           0       0.81      0.68      0.74       193
           1       0.72      0.84      0.77       187

    accuracy                           0.76       380
   macro avg       0.77      0.76      0.76       380
weighted avg       0.77      0.76      0.76       380

